# 01 · Inventario, muestreo y generación de embeddings

Este notebook está diseñado para **Windows nativo**, VS Code y un ambiente administrado con `uv`. Primero inventaría los audios, resume su distribución y selecciona una muestra reproducible. Después genera embeddings de **únicamente los WAV incluidos en el manifiesto**, leyendo siempre los archivos desde sus rutas originales.

No crea enlaces simbólicos, enlaces físicos ni copias de los audios.

Flujo:

1. Configurar rutas y estrategia de muestreo.
2. Inventariar y revisar la distribución por deployment.
3. Seleccionar la muestra y guardar archivos de auditoría.
4. Configurar Perch-Hoplite para leer directamente el manifiesto.
5. Crear o reanudar la base de embeddings.
6. Validar los resultados.

> La carpeta de la base Hoplite debe estar en un disco local estable, preferiblemente SSD. Evite OneDrive, unidades de red y discos que puedan desconectarse durante el proceso.

## 1. Importaciones generales

Este notebook asume que el ambiente del proyecto ya fue creado y que VS Code está usando su kernel. No instala paquetes desde una celda para mantener el ambiente reproducible mediante `pyproject.toml` y `uv.lock`.

In [1]:
from __future__ import annotations

import dataclasses
import hashlib
import json
import os
import platform
import re
from collections.abc import Iterator
from pathlib import Path, PurePosixPath

import pandas as pd
from IPython.display import display

print(f"Python: {platform.python_version()}")
print(f"Sistema: {platform.system()} {platform.release()}")

if platform.system() != "Windows":
    print(
        "ADVERTENCIA: este notebook fue configurado para Windows nativo. "
        "Revise especialmente las rutas antes de continuar."
    )

Python: 3.10.20
Sistema: Windows 10


## 2. Parámetros del usuario

Revise esta celda antes de ejecutar el resto.

Estrategias disponibles:

- `"all"`: usa todos los audios.
- `"random_fraction"`: toma una fracción aleatoria independiente dentro de cada deployment.
- `"alternate_days"`: ordena los días observados dentro de cada deployment y conserva un día sí y otro no.
- `"stratified_month"`: toma una fracción aleatoria dentro de cada combinación deployment × mes.
- `"systematic_files"`: ordena los archivos dentro de cada deployment y conserva uno de cada `file_step`.

Para bases con campañas cortas y varios deployments, `alternate_days` suele ser una opción simple y fácil de explicar. Si el muestreo cubre muchos meses o estaciones, `stratified_month` conserva mejor la representación temporal.

In [2]:
# -----------------------------
# RUTAS Y NOMBRES
# -----------------------------

# Use cadenas raw r"..." para rutas de Windows.
dataset_name = "amistosa_C8"
dataset_base_path = Path(r"G:\AMISTOSA\C8")
dataset_fileglob = "*/*/*.wav"

# La DB se guarda fuera de los audios, por defecto en el perfil del usuario.
# Puede sustituirla, por ejemplo, por Path(r"D:\hoplite_databases").
db_workspace_root = Path(r"D:\hoplite_databases")

# Cambie sample_name cuando quiera crear una selección o DB diferente.
sample_name = "amistosaC8_ad"


# -----------------------------
# ESTRUCTURA DE CARPETAS Y FECHA
# -----------------------------

# group_depth=1 usa la primera carpeta bajo dataset_base_path como deployment.
# Ejemplo: BAJA08/Data/audio.wav -> deployment = BAJA08
group_depth = 1

# Ajuste estas expresiones si la fecha de sus archivos tiene otro formato.
date_regex = r"(?P<date>\d{8})"
date_format = "%Y%m%d"


# -----------------------------
# MUESTREO
# -----------------------------

sampling_strategy = "alternate_days"
random_seed = 2026

# Usados por random_fraction y stratified_month.
sample_fraction = 0.25
min_files_per_stratum = 1

# Usados por alternate_days.
day_step = 2
day_offset = 0

# Usados por systematic_files.
file_step = 2
file_offset = 0


# -----------------------------
# EMBEDDINGS
# -----------------------------

# perch_8 funciona con CPU y genera embeddings de dimensión 1280.
model_choice = "perch_8"
use_file_sharding = True
shard_length_in_seconds = 15
batch_size = 16
audio_worker_threads = min(8, os.cpu_count() or 1)

# True inicia el proceso largo en la sección 8.
run_embeddings = True


# -----------------------------
# RUTAS DERIVADAS
# -----------------------------

dataset_base_path = dataset_base_path.expanduser().resolve()
db_path = (
    db_workspace_root.expanduser().resolve()
    / dataset_base_path.name
    / f"perch_embed_{sample_name}"
)

print(f"Audios: {dataset_base_path}")
print(f"DB:     {db_path}")

Audios: G:\AMISTOSA\C8
DB:     D:\hoplite_databases\C8\perch_embed_amistosaC8_ad


## 3. Funciones de inventario y muestreo

Las funciones se mantienen dentro del notebook para que pueda usarse de forma independiente.

In [3]:
def deployment_from_relative_path(relative_path: str, depth: int = 1) -> str:
    '''Obtiene el deployment desde la estructura relativa de carpetas.'''
    if depth < 1:
        raise ValueError("group_depth debe ser mayor o igual a 1.")

    parts = PurePosixPath(relative_path).parts
    if len(parts) <= depth:
        raise ValueError(
            f"La ruta {relative_path!r} no tiene suficientes niveles para "
            f"group_depth={depth}."
        )
    return parts[depth - 1]


def extract_date_from_filename(
    filename: str,
    pattern: str,
    fmt: str,
) -> pd.Timestamp:
    '''Extrae la fecha desde el nombre del archivo; retorna NaT si no coincide.'''
    match = re.search(pattern, filename)
    if match is None:
        return pd.NaT

    value = match.groupdict().get("date")
    if value is None:
        value = match.group(0)
    return pd.to_datetime(value, format=fmt, errors="coerce")


def build_audio_inventory(
    dataset_root: Path,
    file_glob: str,
    depth: int,
    pattern: str,
    fmt: str,
) -> pd.DataFrame:
    '''Escanea los audios y construye un inventario auditable.'''
    dataset_root = dataset_root.expanduser().resolve()
    if not dataset_root.is_dir():
        raise FileNotFoundError(f"No existe la carpeta: {dataset_root}")

    paths = sorted(path for path in dataset_root.glob(file_glob) if path.is_file())
    rows = []
    for path in paths:
        relative_path = path.relative_to(dataset_root).as_posix()
        rows.append(
            {
                "relative_path": relative_path,
                "deployment": deployment_from_relative_path(relative_path, depth),
                "filename": path.name,
                "date": extract_date_from_filename(path.name, pattern, fmt),
                "size_bytes": path.stat().st_size,
            }
        )

    inventory = pd.DataFrame.from_records(rows)
    if inventory.empty:
        return pd.DataFrame(
            columns=[
                "relative_path",
                "deployment",
                "filename",
                "date",
                "year_month",
                "size_bytes",
            ]
        )

    inventory["date"] = pd.to_datetime(inventory["date"], errors="coerce")
    inventory["year_month"] = inventory["date"].dt.to_period("M").astype("string")
    return inventory.sort_values("relative_path").reset_index(drop=True)


def _sample_group_fraction(
    group: pd.DataFrame,
    fraction: float,
    minimum: int,
    rng,
) -> pd.DataFrame:
    if not 0 < fraction <= 1:
        raise ValueError("sample_fraction debe estar en el intervalo (0, 1].")
    if minimum < 1:
        raise ValueError("min_files_per_stratum debe ser al menos 1.")

    n = min(len(group), max(minimum, int(round(len(group) * fraction))))
    state = int(rng.integers(0, 2**32 - 1))
    return group.sample(n=n, random_state=state)


def select_audio_sample(
    inventory: pd.DataFrame,
    strategy: str,
    fraction: float,
    seed: int,
    minimum: int,
    day_step: int,
    day_offset: int,
    file_step: int,
    file_offset: int,
) -> pd.DataFrame:
    '''Selecciona audios sin mover, copiar ni enlazar archivos.'''
    valid_strategies = {
        "all",
        "random_fraction",
        "alternate_days",
        "stratified_month",
        "systematic_files",
    }
    if strategy not in valid_strategies:
        raise ValueError(
            f"Estrategia desconocida: {strategy!r}. Use una de {sorted(valid_strategies)}."
        )

    if strategy == "all":
        selected = inventory.copy()

    elif strategy == "random_fraction":
        import numpy as np

        rng = np.random.default_rng(seed)
        pieces = [
            _sample_group_fraction(group, fraction, minimum, rng)
            for _, group in inventory.groupby("deployment", sort=True)
        ]
        selected = pd.concat(pieces, ignore_index=True)

    elif strategy == "stratified_month":
        if inventory["date"].isna().any():
            raise ValueError(
                "stratified_month requiere una fecha reconocida en todos los audios. "
                "Revise date_regex y date_format."
            )
        import numpy as np

        rng = np.random.default_rng(seed)
        pieces = [
            _sample_group_fraction(group, fraction, minimum, rng)
            for _, group in inventory.groupby(
                ["deployment", "year_month"], sort=True, observed=True
            )
        ]
        selected = pd.concat(pieces, ignore_index=True)

    elif strategy == "alternate_days":
        if day_step < 1:
            raise ValueError("day_step debe ser mayor o igual a 1.")
        if not 0 <= day_offset < day_step:
            raise ValueError("day_offset debe cumplir 0 <= day_offset < day_step.")
        if inventory["date"].isna().any():
            raise ValueError(
                "alternate_days requiere una fecha reconocida en todos los audios. "
                "Revise date_regex y date_format."
            )

        pieces = []
        for _, group in inventory.groupby("deployment", sort=True):
            normalized_dates = group["date"].dt.normalize()
            observed_days = sorted(normalized_dates.unique())
            keep_days = set(observed_days[day_offset::day_step])
            pieces.append(group.loc[normalized_dates.isin(keep_days)])
        selected = pd.concat(pieces, ignore_index=True)

    else:  # systematic_files
        if file_step < 1:
            raise ValueError("file_step debe ser mayor o igual a 1.")
        if not 0 <= file_offset < file_step:
            raise ValueError("file_offset debe cumplir 0 <= file_offset < file_step.")

        pieces = []
        for _, group in inventory.groupby("deployment", sort=True):
            ordered = group.sort_values(["date", "relative_path"], na_position="last")
            pieces.append(ordered.iloc[file_offset::file_step])
        selected = pd.concat(pieces, ignore_index=True)

    return (
        selected.drop_duplicates("relative_path")
        .sort_values("relative_path")
        .reset_index(drop=True)
    )


def selection_fingerprint(selected: pd.DataFrame) -> str:
    '''Huella reproducible del conjunto exacto de rutas seleccionadas.'''
    relative_paths = sorted(selected["relative_path"].astype(str).tolist())
    payload = "\n".join(relative_paths).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

## 4. Inventariar y revisar la base de audios

Esta etapa no modifica la carpeta original.

In [4]:
inventory = build_audio_inventory(
    dataset_root=dataset_base_path,
    file_glob=dataset_fileglob,
    depth=group_depth,
    pattern=date_regex,
    fmt=date_format,
)

if inventory.empty:
    raise FileNotFoundError(
        "No se encontraron audios. Revise dataset_base_path y dataset_fileglob."
    )

distribution_by_deployment = (
    inventory.groupby("deployment", dropna=False)
    .agg(
        n_audio=("relative_path", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        n_days=("date", lambda x: x.dt.normalize().nunique()),
        n_months=("year_month", "nunique"),
        total_gb=("size_bytes", lambda x: x.sum() / 1024**3),
    )
    .reset_index()
)

print(f"Audios encontrados: {len(inventory):,}")
print(f"Deployments/carpetas: {inventory['deployment'].nunique():,}")
print(f"Audios sin fecha reconocida: {inventory['date'].isna().sum():,}")
print(f"Tamaño total aproximado: {inventory['size_bytes'].sum() / 1024**3:,.2f} GB")
display(distribution_by_deployment)

Audios encontrados: 54,047
Deployments/carpetas: 52
Audios sin fecha reconocida: 0
Tamaño total aproximado: 2,663.48 GB


,deployment,n_audio,first_date,last_date,n_days,n_months,total_gb
0,ACA12_T75,2008,2024-06-27,2024-08-22,57,3,98.969068
1,ALM15_T111 - R,1008,2024-05-28,2024-07-21,28,3,49.668310
2,BMA07_A_T103,1205,2024-05-29,2024-07-01,34,3,59.391298
3,BMA07_B_T163,824,2024-07-01,2024-07-24,24,1,40.612805
4,BOA06_A_T128,1204,2024-05-29,2024-07-01,34,3,59.342011
5,BOA06_B_T151,814,2024-07-01,2024-07-24,24,1,40.119931
6,BQM18_T85,1008,2024-06-26,2024-08-19,28,3,49.668310
7,C1M09_T74,970,2024-06-28,2024-08-19,27,3,47.795893
8,C2M13_T164,108,2024-06-27,2024-07-01,3,2,5.321605
9,CAA04_A_T94,1205,2024-05-29,2024-07-02,35,3,59.391298


## 5. Seleccionar la muestra

La tabla permite revisar cuántos audios y días se conservarán en cada deployment antes de iniciar el proceso largo.

In [5]:
selected_audio = select_audio_sample(
    inventory=inventory,
    strategy=sampling_strategy,
    fraction=sample_fraction,
    seed=random_seed,
    minimum=min_files_per_stratum,
    day_step=day_step,
    day_offset=day_offset,
    file_step=file_step,
    file_offset=file_offset,
)

if selected_audio.empty:
    raise ValueError("La estrategia produjo una muestra vacía.")

selected_distribution = (
    selected_audio.groupby("deployment", dropna=False)
    .agg(
        selected_audio=("relative_path", "size"),
        selected_days=("date", lambda x: x.dt.normalize().nunique()),
        first_date=("date", "min"),
        last_date=("date", "max"),
        selected_gb=("size_bytes", lambda x: x.sum() / 1024**3),
    )
    .reset_index()
)

available_counts = distribution_by_deployment.set_index("deployment")["n_audio"]
selected_distribution["available_audio"] = selected_distribution["deployment"].map(
    available_counts
)
selected_distribution["selected_fraction"] = (
    selected_distribution["selected_audio"]
    / selected_distribution["available_audio"]
)

print(
    f"Estrategia: {sampling_strategy} | "
    f"seleccionados: {len(selected_audio):,} de {len(inventory):,} "
    f"({len(selected_audio) / len(inventory):.1%})"
)
display(selected_distribution)

Estrategia: alternate_days | seleccionados: 27,146 de 54,047 (50.2%)


,deployment,selected_audio,selected_days,first_date,last_date,selected_gb,available_audio,selected_fraction
0,ACA12_T75,1000,29,2024-06-27,2024-08-22,49.287384,2008,0.498008
1,ALM15_T111 - R,504,14,2024-05-28,2024-07-19,24.834155,1008,0.500000
2,BMA07_A_T103,600,17,2024-05-29,2024-06-30,29.572431,1205,0.497925
3,BMA07_B_T163,403,12,2024-07-01,2024-07-23,19.862816,824,0.489078
4,BOA06_A_T128,597,17,2024-05-29,2024-06-30,29.424568,1204,0.495847
5,BOA06_B_T151,401,12,2024-07-01,2024-07-23,19.764241,814,0.492629
6,BQM18_T85,504,14,2024-06-26,2024-08-17,24.834155,1008,0.500000
7,C1M09_T74,502,14,2024-06-28,2024-08-19,24.735607,970,0.517526
8,C2M13_T164,72,2,2024-06-27,2024-07-01,3.547736,108,0.666667
9,CAA04_A_T94,593,18,2024-05-29,2024-07-02,29.227419,1205,0.492116


## 6. Verificar rutas originales y guardar auditoría

La DB queda ligada a la huella SHA-256 de la selección. Si cambia el manifiesto, el notebook se detiene y solicita usar otro `sample_name`; así no se mezclan embeddings de muestras diferentes.

In [6]:
selected_audio = selected_audio.copy()
selected_audio["absolute_path"] = selected_audio["relative_path"].map(
    lambda relative_path: dataset_base_path / Path(PurePosixPath(relative_path))
)

missing_mask = ~selected_audio["absolute_path"].map(Path.is_file)
missing_audio = selected_audio.loc[missing_mask]

print(f"Audios seleccionados: {len(selected_audio):,}")
print(f"Audios existentes:    {len(selected_audio) - len(missing_audio):,}")
print(f"Audios faltantes:     {len(missing_audio):,}")

if not missing_audio.empty:
    display(missing_audio[["deployment", "relative_path"]].head(20))
    raise FileNotFoundError(
        "Algunos audios seleccionados ya no existen en su ubicación original."
    )

db_path.mkdir(parents=True, exist_ok=True)
audit_path = db_path / "sampling_audit"
audit_path.mkdir(parents=True, exist_ok=True)

fingerprint = selection_fingerprint(selected_audio)
fingerprint_path = audit_path / "selection_sha256.txt"
processing_payload = {
    "selection_sha256": fingerprint,
    "dataset_name": dataset_name,
    "model_choice": model_choice,
    "use_file_sharding": use_file_sharding,
    "shard_length_in_seconds": shard_length_in_seconds,
}
processing_fingerprint = hashlib.sha256(
    json.dumps(processing_payload, sort_keys=True).encode("utf-8")
).hexdigest()
processing_fingerprint_path = audit_path / "processing_sha256.txt"
db_artifacts_exist = any(db_path.glob("hoplite.sqlite*")) or (
    db_path / "usearch.index"
).exists()

if fingerprint_path.exists():
    previous_fingerprint = fingerprint_path.read_text(encoding="utf-8").strip()
    if previous_fingerprint and previous_fingerprint != fingerprint:
        raise ValueError(
            "La selección cambió, pero esta DB pertenece a otro manifiesto. "
            "Cambie sample_name para crear una DB nueva."
        )
elif db_artifacts_exist:
    raise ValueError(
        "La DB ya contiene archivos, pero no existe una huella de muestreo para "
        "verificar su origen. Use otro sample_name."
    )

if processing_fingerprint_path.exists():
    previous_processing_fingerprint = processing_fingerprint_path.read_text(
        encoding="utf-8"
    ).strip()
    if (
        previous_processing_fingerprint
        and previous_processing_fingerprint != processing_fingerprint
    ):
        raise ValueError(
            "Cambió el modelo o la configuración de sharding asociada a esta "
            "DB. Cambie sample_name para crear una DB nueva."
        )

# El manifiesto guarda rutas relativas para que siga siendo legible y portable.
manifest_to_save = selected_audio.drop(columns="absolute_path")
manifest_to_save.to_csv(audit_path / "sample_manifest.csv", index=False)
selected_distribution.to_csv(
    audit_path / "sample_distribution.csv", index=False
)
distribution_by_deployment.to_csv(
    audit_path / "inventory_distribution.csv", index=False
)
fingerprint_path.write_text(fingerprint + "\n", encoding="utf-8")
processing_fingerprint_path.write_text(
    processing_fingerprint + "\n", encoding="utf-8"
)

sampling_config = {
    "dataset_name": dataset_name,
    "dataset_base_path": str(dataset_base_path),
    "dataset_fileglob": dataset_fileglob,
    "sample_name": sample_name,
    "sampling_strategy": sampling_strategy,
    "sample_fraction": sample_fraction,
    "random_seed": random_seed,
    "min_files_per_stratum": min_files_per_stratum,
    "day_step": day_step,
    "day_offset": day_offset,
    "file_step": file_step,
    "file_offset": file_offset,
    "group_depth": group_depth,
    "date_regex": date_regex,
    "date_format": date_format,
    "model_choice": model_choice,
    "selection_sha256": fingerprint,
}
(audit_path / "sampling_config.json").write_text(
    json.dumps(sampling_config, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print(f"DB path:    {db_path}")
print(f"Manifiesto: {audit_path / 'sample_manifest.csv'}")
print(f"SHA-256:    {fingerprint}")
print(f"Proceso:    {processing_fingerprint}")
print("No se crearon enlaces ni copias de los WAV.")

Audios seleccionados: 27,146
Audios existentes:    27,146
Audios faltantes:     0
DB path:    D:\hoplite_databases\C8\perch_embed_amistosaC8_ad
Manifiesto: D:\hoplite_databases\C8\perch_embed_amistosaC8_ad\sampling_audit\sample_manifest.csv
SHA-256:    976d8cb1f6c2e2c86a62b0df074f29620986c87070cf77d53a7521bb0937331d
Proceso:    98236285ffa58fb1bd61e778f377bb5ea6d037150066f14963aacae20b7bdd01
No se crearon enlaces ni copias de los WAV.


## 7. Configurar Perch-Hoplite para leer el manifiesto

`ManifestAudioSources` conserva un único proyecto y los deployments derivados de las rutas relativas. Solo reemplaza el descubrimiento por `glob`; el resto del procesamiento sigue a cargo de `EmbedWorker`.

La implementación también conserva el manifiesto al reanudar una DB. Esta protección es importante porque Hoplite guarda la configuración general del dataset, pero no conoce por sí mismo la selección exacta.

In [7]:
import importlib.metadata

from etils import epath
from perch_hoplite.agile import colab_utils, embed, source_info
from tqdm.auto import tqdm

try:
    print("perch-hoplite:", importlib.metadata.version("perch-hoplite"))
except importlib.metadata.PackageNotFoundError:
    print("perch-hoplite instalado desde una fuente sin metadatos de versión.")


@dataclasses.dataclass
class ManifestAudioSources(source_info.AudioSources):
    '''AudioSources que itera únicamente las rutas relativas del manifiesto.'''

    manifest_relative_paths: tuple[str, ...] = dataclasses.field(
        default_factory=tuple,
        repr=False,
        hash=False,
    )

    def __post_init__(self):
        super().__post_init__()
        if len(self.audio_globs) != 1:
            raise ValueError(
                "ManifestAudioSources requiere exactamente un AudioSourceConfig."
            )
        if not self.manifest_relative_paths:
            raise ValueError("El manifiesto no puede estar vacío.")
        if len(set(self.manifest_relative_paths)) != len(
            self.manifest_relative_paths
        ):
            raise ValueError("El manifiesto contiene rutas duplicadas.")

    def merge_update(
        self,
        other: source_info.AudioSources,
    ) -> "ManifestAudioSources":
        '''Valida una DB existente sin perder el iterador del manifiesto.'''
        if len(other.audio_globs) != 1:
            raise ValueError(
                "La DB contiene una configuración distinta a la esperada. "
                "Use otro sample_name."
            )

        current = self.audio_globs[0]
        previous = other.audio_globs[0]
        if not current.is_compatible(previous):
            raise ValueError(
                "La configuración guardada en la DB no es compatible con la "
                "configuración actual. Use otro sample_name."
            )

        # Se conserva la ruta actual, útil si el dataset fue movido.
        return type(self)(
            audio_globs=self.audio_globs,
            manifest_relative_paths=self.manifest_relative_paths,
        )

    def iterate_all_sources(
        self,
        target_dataset_name: str | None = None,
    ) -> Iterator[source_info.SourceId]:
        '''Genera SourceId solo para los archivos seleccionados.'''
        config = self.audio_globs[0]
        if (
            target_dataset_name is not None
            and config.dataset_name != target_dataset_name
        ):
            return

        base_path = Path(config.base_path).expanduser().resolve()
        shard_len_s = config.shard_len_s
        max_shards_per_file = config.max_shards_per_file

        for relative_path in tqdm(
            self.manifest_relative_paths,
            desc="Audios del manifiesto",
        ):
            file_id = PurePosixPath(relative_path).as_posix()
            filepath = base_path / Path(PurePosixPath(file_id))
            if not filepath.is_file():
                raise FileNotFoundError(f"No existe el audio: {filepath}")

            filepath_for_hoplite = epath.Path(filepath.as_posix())
            audio_len_s, sample_rate_hz = (
                self._get_audio_len_s_and_sample_rate_hz(filepath_for_hoplite)
            )

            if shard_len_s is None:
                yield source_info.SourceId(
                    dataset_name=config.dataset_name,
                    file_id=file_id,
                    offset_s=0,
                    shard_len_s=-1,
                    filepath=filepath.as_posix(),
                    sample_rate_hz=sample_rate_hz,
                )
                continue

            if audio_len_s <= 0:
                continue

            shard_num = 0
            while max_shards_per_file is None or shard_num < max_shards_per_file:
                offset_s = shard_num * shard_len_s
                if offset_s >= audio_len_s:
                    break
                if (
                    offset_s + shard_len_s > audio_len_s
                    and audio_len_s - offset_s < config.min_audio_len_s
                ):
                    break

                yield source_info.SourceId(
                    dataset_name=config.dataset_name,
                    file_id=file_id,
                    offset_s=offset_s,
                    shard_len_s=shard_len_s,
                    filepath=filepath.as_posix(),
                    sample_rate_hz=sample_rate_hz,
                )
                shard_num += 1


audio_config = source_info.AudioSourceConfig(
    dataset_name=dataset_name,
    base_path=dataset_base_path.as_posix(),
    # Se conserva como descripción general; el iterador usa el manifiesto.
    file_glob=dataset_fileglob,
    min_audio_len_s=1.0,
    target_sample_rate_hz=-2,
    shard_len_s=(
        float(shard_length_in_seconds) if use_file_sharding else None
    ),
)

manifest_relative_paths = tuple(
    selected_audio["relative_path"].astype(str).sort_values().tolist()
)
manifest_audio_sources = ManifestAudioSources(
    audio_globs=(audio_config,),
    manifest_relative_paths=manifest_relative_paths,
)

configs = colab_utils.load_configs(
    audio_sources=manifest_audio_sources,
    db_path=db_path.as_posix(),
    model_config_key=model_choice,
    db_key="sqlite_usearch",
)

print(f"Proyecto Hoplite: {dataset_name}")
print(f"Rutas en el manifiesto: {len(manifest_relative_paths):,}")
print(f"Modelo: {model_choice}")
print(f"Dimensión: {configs.model_config.embedding_dim}")

perch-hoplite: 1.0.2
Proyecto Hoplite: amistosa_C8
Rutas en el manifiesto: 27,146
Modelo: perch_8
Dimensión: 1280


## 8. Abrir la DB y generar embeddings

Una DB incompleta que tenga grabaciones registradas pero cero embeddings se considera insegura para reanudar con la lógica estándar de Hoplite. En ese caso, use un `sample_name` nuevo. Cuando ya existen embeddings, `handle_duplicates="skip"` permite continuar sin duplicar las ventanas terminadas.

In [8]:
db = configs.db_config.load_db()

num_embeddings_before = db.count_embeddings()
num_recordings_before = len(db.get_all_recordings())
num_deployments_before = len(db.get_all_deployments())

print(f"Embeddings existentes:  {num_embeddings_before:,}")
print(f"Grabaciones existentes: {num_recordings_before:,}")
print(f"Deployments existentes: {num_deployments_before:,}")

if num_recordings_before > 0 and num_embeddings_before == 0:
    raise RuntimeError(
        "La DB parece corresponder a una ejecución fallida: contiene grabaciones "
        "pero cero embeddings. Cambie sample_name para iniciar una DB limpia."
    )

Embeddings existentes:  0
Grabaciones existentes: 0
Deployments existentes: 0


In [9]:
# Este proceso puede tardar varias horas

worker = embed.EmbedWorker(
    audio_sources=manifest_audio_sources,
    db=db,
    model_config=configs.model_config,
    audio_worker_threads=audio_worker_threads,
    )


worker.process_all(
    target_dataset_name=dataset_name,
    batch_size=batch_size
    )

print(f"Embeddings totales: {db.count_embeddings():,}")



Adding deployments...


Audios del manifiesto:   0%|          | 0/27146 [00:00<?, ?it/s]


Adding recordings...


Audios del manifiesto:   0%|          | 0/27146 [00:00<?, ?it/s]


Adding annotations...

Embedding audio...


Audios del manifiesto:   0%|          | 0/27146 [00:00<?, ?it/s]

Embeddings totales: 1,601,614


## 9. Validación final

La cantidad de grabaciones debe coincidir con la cantidad de audios seleccionados. La cantidad de embeddings será mayor porque cada archivo genera varias ventanas temporales.

In [10]:
if run_embeddings:
    db.commit()

    final_recordings = db.get_all_recordings()
    final_deployments = db.get_all_deployments()
    final_embeddings = db.count_embeddings()

    print(f"Audios seleccionados:    {len(selected_audio):,}")
    print(f"Grabaciones en la DB:    {len(final_recordings):,}")
    print(f"Deployments en la DB:    {len(final_deployments):,}")
    print(f"Embeddings en la DB:     {final_embeddings:,}")

    if len(final_recordings) != len(selected_audio):
        raise AssertionError(
            "La cantidad de grabaciones de la DB no coincide con el manifiesto. "
            "Revise los mensajes de carga de audio y no reutilice esta DB con "
            "otra selección."
        )

    validation_summary = {
        "selected_audio": len(selected_audio),
        "database_recordings": len(final_recordings),
        "database_deployments": len(final_deployments),
        "database_embeddings": final_embeddings,
        "selection_sha256": fingerprint,
        "status": "ok",
    }
    (audit_path / "embedding_validation.json").write_text(
        json.dumps(validation_summary, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    print("Validación completada correctamente.")
else:
    print("No hay resultados que validar porque run_embeddings=False.")

Audios seleccionados:    27,146
Grabaciones en la DB:    27,146
Deployments en la DB:    52
Embeddings en la DB:     1,601,614
Validación completada correctamente.


## 10. Cerrar la DB al terminar

Ejecute esta celda cuando ya no vaya a consultar la base en este notebook. Esto libera conexiones antes de abrir la misma DB desde otro notebook o proceso.

In [11]:
db.commit()

close_method = getattr(db, "close", None)
if callable(close_method):
    close_method()
    print("DB cerrada.")
else:
    print(
        "La versión instalada de Hoplite no expone db.close(). "
        "El commit fue realizado; cierre el kernel antes de abrir esta DB "
        "desde otro proceso."
    )

La versión instalada de Hoplite no expone db.close(). El commit fue realizado; cierre el kernel antes de abrir esta DB desde otro proceso.
